# Museum Guide — a Basic RAG System

A tiny Retrieval-Augmented Generation assistant for the **Raclin Murphy Museum of Art**, built on the exact Week 8 pattern: embed your documents into a vector database, search it for the passages closest to a question, and hand those passages to the model to answer with citations.

The only thing that changed from the Week 8 code is the **source of the documents**: instead of copying a pre-built database from Drive, we build one from your spreadsheet. Run the cells top to bottom.

**Before you start:** add your `OPENAI_API_KEY` in Colab Secrets (the key icon on the left).

## Step 0: Setup

One install cell (LangChain, its OpenAI and Chroma connectors, Chroma itself, and `openpyxl` for reading the spreadsheet), at the versions we tested. If pip ends with a red `ERROR` about packages we never import, the install still worked.

In [1]:
# pinned: these exact versions were tested together on 2026-10-11
# (LangChain itself, its OpenAI and Chroma connectors, and the Chroma vector database; openpyxl reads .xlsx)
!pip install -q langchain==1.4.3 langchain-core==1.6.6 langchain-openai==1.6.6 langchain-chroma==1.1.0 chromadb==1.5.9 openpyxl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
t

In [3]:
import os
import pandas as pd
from google.colab import userdata

# Your OpenAI key lives in Colab Secrets (the key icon). LangChain looks for it in this environment variable.
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# The LangChain pieces we use:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings          # OpenAI's chat model and embedding model
from langchain_chroma import Chroma                                # Chroma, our vector database
from langchain_core.documents import Document                      # one record = one Document
from langchain_core.messages import SystemMessage, HumanMessage    # the message types in a chat

# temperature=0 means the model picks its most likely words, so everyone's output is close to the same.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)                 # writes the replies
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")        # turns text into vectors for search

## Step 1: Upload your spreadsheet

Run this cell and pick `Raclin_Murphy_Museum_Galleries_Dataset.xlsx` from your computer. It lands in this Colab machine so the next cell can read it. (You re-upload each time Colab restarts; to skip that, mount Google Drive instead and point `SPREADSHEET` below at the file's Drive path.)

In [4]:
from google.colab import files
uploaded = files.upload()      # click "Choose Files" and select the .xlsx

Saving Raclin_Murphy_Museum_Galleries_Dataset.xlsx to Raclin_Murphy_Museum_Galleries_Dataset.xlsx


## Step 2: Build the vector database from the spreadsheet

This is the one new piece. We turn **each row into one Document**: the text the model reads, plus a `title` and `locator` for citations. Then we embed them all into a fresh Chroma database. Each sheet has two banner rows and a blank row before its real column headers, which is why `header=3`.

To point this notebook at different data later, change `SPREADSHEET` and the `SHEETS` list — nothing else.

In [5]:
SPREADSHEET = "Raclin_Murphy_Museum_Galleries_Dataset.xlsx"   # the file you uploaded above

# Which sheets to load, and which columns to use as the (title, locator) label the model cites.
SHEETS = [
    {"sheet": "Artworks",  "header": 3, "title_col": "Title",   "locator_cols": ["Gallery (assigned)", "Accession No."]},
    {"sheet": "Galleries", "header": 3, "title_col": "Gallery", "locator_cols": ["Level"]},
]

def row_to_text(row, cols):
    # one record -> one readable passage: every non-empty "Column: value" line
    return "\n".join(f"{c}: {row[c]}" for c in cols
                     if c != "#" and pd.notna(row[c]) and str(row[c]).strip())

docs = []
for spec in SHEETS:
    df = pd.read_excel(SPREADSHEET, sheet_name=spec["sheet"], header=spec["header"]).dropna(how="all")
    cols = list(df.columns)
    for _, row in df.iterrows():
        title = str(row.get(spec["title_col"], "")).strip()
        if not title or title.lower() == "nan":               # skip blank rows
            continue
        locator = ", ".join(str(row[c]).strip() for c in spec["locator_cols"]
                            if c in df.columns and pd.notna(row[c]) and str(row[c]).strip())
        docs.append(Document(page_content=row_to_text(row, cols),
                             metadata={"title": title, "locator": locator or spec["sheet"]}))

print(f"Built {len(docs)} records from {SPREADSHEET}.")

# Embed every record into a fresh Chroma database. No persist_directory, so it lives in memory and is
# rebuilt each run; add persist_directory="/content/museum_db" to save it between runs.
collection = Chroma.from_documents(docs, embedding=embeddings, collection_name="museum")
print(f"collection database: {collection._collection.count():,} records ready to search")

Built 279 records from Raclin_Murphy_Museum_Galleries_Dataset.xlsx.
collection database: 279 records ready to search


## Step 3: Search and answer (the RAG core)

The same pattern as Week 8's `search_readings`, pointed at the museum: `search_collection` finds the 4 records closest in meaning to a question and labels each with its title and locator, so every answer can carry citations. Then one `llm.invoke` call stitches together the instructions, the retrieved records, and the question.

In [6]:
def search_collection(question, k=4):
    # Embed the question and find the k records closest to it in meaning (the "R" in RAG).
    results = collection.similarity_search(question, k=k)
    # Label every record with its title and locator so the model can cite it. Records are separated by blank lines.
    return "\n\n".join(f"[{d.metadata['title']}, {d.metadata['locator']}] {d.page_content}" for d in results)

# The system message is the assistant's standing instructions. It goes first in every call.
SYSTEM = SystemMessage(
    "You are a gallery guide for the Raclin Murphy Museum of Art at the University of Notre Dame. Answer using "
    "only the collection records you are given, and cite them as (title, locator). If the records don't cover "
    "it, say so. Keep replies to three sentences.")

# 1. The question.
question = "What African artworks are in the collection?"
# 2. One call: instructions + retrieved records + the question, as ONE list of messages.
reply = llm.invoke([SYSTEM, SystemMessage("Collection records:\n" + search_collection(question)), HumanMessage(question)])
print(reply.content)                 # 3. the answer, with citations

The collection includes several notable African artworks such as the "Mbele a Lulendo, Sword of Power," created by an unrecorded Kongo artist in the 18th to 19th century, and a "Memorial Figure." Additionally, there is a "Catalogue of Images of Antiquities, Musee Boulaq, Cairo," dating from 1870-1880 (Mbele a Lulendo, Sword of Power, 2009.063.016; Memorial Figure, 2017.025.995; Catalogue of Images of Antiquities, Musee Boulaq, Cairo, 1994.030.033).


## Step 4: Ask your own questions

`ask()` wraps the same three-part call so you can fire off questions in a loop. Edit the list, or call `ask("...")` yourself.

In [7]:
def ask(question):
    reply = llm.invoke([SYSTEM, SystemMessage("Collection records:\n" + search_collection(question)), HumanMessage(question)])
    return reply.content

for q in ["What did Zhang Huan use to create his 1965 Last Supper artwork?",
          "Who created Wall of Light Black in 1998?",
          "What colors are used in the painting Spring by Natalia Goncharova?",
          "What artifacts are used in the White Rit sculture created by Vanessa German?",
          "In what year did Francesco de Mura create Bacchus and Ceres?",
          "Who gifted the 1786 painting Marguerite Baudard de Saint James to the Raclin Murphy?",
          "Why is The Artist’s Studio, a Sketch by Jean-Léon Gérôme an important piece in the Raclin Murphy Museum?",
          "Have any pieces of art been donated as past of the Bernard Norling and Mary T. Norling Endowment for 18th- and 19th-Century Sculpture?"]:
    print("YOU: ", q)
    print("GUIDE:", ask(q), "\n")

YOU:  What did Zhang Huan use to create his 1965 Last Supper artwork?
GUIDE: Zhang Huan created his monumental gray version of Christ's Last Supper, titled "Ash Banquet #4," using incense ash and pigment collected from burnt incense rods at Buddhist temples (Ash Banquet #4, International Modern and Contemporary Art). 

YOU:  Who created Wall of Light Black in 1998?
GUIDE: Wall of Light Black was created by Sean Scully, an Irish-American artist, in 1998 (Wall of Light Black, International Modern and Contemporary Art). 

YOU:  What colors are used in the painting Spring by Natalia Goncharova?
GUIDE: I don't have information on a painting titled "Spring" by Natalia Goncharova in the collection records provided. Please check other sources for details on that artwork. 

YOU:  What artifacts are used in the White Rit sculture created by Vanessa German?
GUIDE: I'm sorry, but the collection records provided do not include any information about a sculpture titled "White Rit" by Vanessa German. 

In [8]:
def ask(question):
    reply = llm.invoke([SYSTEM, SystemMessage("Collection records:\n" + search_collection(question)), HumanMessage(question)])
    return reply.content

for q in ["Where can I find the chapel in the Raclin Murphy?",
          "How do I get to Art Studio B in the Raclin Murphy Museum from the Welcome Desk?",
          "What are the current temporary exhibitions at the Raclin Murphy Museum?",
          "When will Mimmo Paladino: Present into Past be at the Raclin Murphy Museum until?",
          "How big is Mimmo Paladino: Present into Past?",
          "What upcoming exhibition is coming to Raclin Murphy in February?",
          "Has the Raclin Murphy Museum ever had an Emilio Sanchez exhibit?",]:
    print("YOU: ", q)
    print("GUIDE:", ask(q), "\n")

YOU:  Where can I find the chapel in the Raclin Murphy?
GUIDE: The Chapel of the Holy Child of Atocha is part of the Art of the Spanish Americas collection, but it is currently not on view (Chapel of the Holy Child of Atocha, La Manga, New Mexico, 1996.043.002). You can find more information about it on the museum's Marble page [here](https://marble.nd.edu/item/1996.043.002). 

YOU:  How do I get to Art Studio B in the Raclin Murphy Museum from the Welcome Desk?
GUIDE: I'm sorry, but the collection records do not provide information on navigating the Raclin Murphy Museum or directions to specific locations like Art Studio B. Please consult a museum map or ask a staff member at the Welcome Desk for assistance. 

YOU:  What are the current temporary exhibitions at the Raclin Murphy Museum?
GUIDE: The collection records do not provide information about current temporary exhibitions at the Raclin Murphy Museum of Art. Please check the museum's official website or contact them directly for 